# v7 Inference Notebook — PM2.5 Forecasting

Loads the trained v7 checkpoint and runs inference on the test set.  
**No training required.** Just set `CHECKPOINT_PATH` and run all cells.

**Requirements:** `v7_final.pt` checkpoint in the path below.


## 0. Imports & Config

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.cluster import KMeans
from scipy.stats import skew as scipy_skew
from statsmodels.tsa.seasonal import STL
import gc, warnings

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── PATHS — set these before running ─────────────────────────────
CHECKPOINT_PATH = Path('/kaggle/working/v7_final.pt')   # ← path to checkpoint
DATA_ROOT       = Path('/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw')
TEST_DIR        = Path('/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in')
OUTPUT          = Path('/kaggle/working'); OUTPUT.mkdir(exist_ok=True)
LAT_LON         = DATA_ROOT / 'lat_long.npy'
ALL_MONTHS      = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']

# ── v7 CONFIG (must match training) ──────────────────────────────
LOOKBACK = 10; FORECAST = 16; GRID_H = 140; GRID_W = 124
TRAIN_HOURS = 600

MET_VARS   = ['pblh','u10','v10','rain','t2','q2','psfc','swdown']
EMIS_VARS  = ['SO2','NMVOC_e','PM25','NOx','NH3','NMVOC_finn','bio']
ALL_FEAT   = ['cpm25'] + MET_VARS + EMIS_VARS
LOG_CANDIDATES = {'cpm25','rain','SO2','NMVOC_e','PM25','NOx','NH3','NMVOC_finn','bio','swdown','pblh'}
UNIT_SCALE = {f: 1e9 for f in ['SO2','NMVOC_e','PM25','NOx','NH3','NMVOC_finn','bio']}

K_CLUSTERS    = 16
N_CHANNELS    = len(ALL_FEAT) + 4   # 20
BASE_CHANNELS = 24
CONVLSTM_DIM  = 48
N_LEVELS      = 2
LSTM_LAYERS   = 1

print(f'Config: K={K_CLUSTERS} | N_CHANNELS={N_CHANNELS} | base={BASE_CHANNELS} | dim={CONVLSTM_DIM}')
assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
print(f'Checkpoint: {CHECKPOINT_PATH} ✓')


## 1. Spatial Grid + Clusters
*Recomputes the same K=16 clustering used during training.*

In [ ]:
ll = np.load(LAT_LON)
lat_grid = ll[..., 0]; lon_grid = ll[..., 1]

def load_feature(month, feat, root=DATA_ROOT):
    p = root/month/f'{feat}.npy'
    return np.load(p).astype(np.float32) if p.exists() else None

def detect_episodes_grid(arr, period=24, threshold_mult=1.0, subsample=4):
    T,H,W = arr.shape; ep = np.zeros((T,H,W),dtype=bool)
    for ri in range(0,H,subsample):
        for ci in range(0,W,subsample):
            ts = arr[:,ri,ci].astype(float)
            if ts.std()<0.5 or len(ts)<2*period: continue
            try:
                from statsmodels.tsa.seasonal import STL
                res=STL(ts,period=period,robust=True).fit()
                e=res.resid>res.resid.std()*threshold_mult
                ep[:,ri:min(ri+subsample,H),ci:min(ci+subsample,W)]=e[:,None,None]
            except: continue
    return ep

print('Building K=16 clusters (must match training)...')
cl_feats=[]; pm25_mean=np.zeros((GRID_H,GRID_W),np.float32)
pm25_var =np.zeros((GRID_H,GRID_W),np.float32); ep_freq=np.zeros((GRID_H,GRID_W),np.float32)
for month in ALL_MONTHS:
    d=load_feature(month,'cpm25')
    cl_feats.append(np.log1p(d).mean(0).flatten())
    pm25_mean+=d.mean(0); pm25_var+=d.var(0)
    ep=detect_episodes_grid(d[:TRAIN_HOURS],subsample=4); ep_freq+=ep.mean(0)
    del d,ep; gc.collect()
pm25_mean/=len(ALL_MONTHS); pm25_var/=len(ALL_MONTHS); ep_freq/=len(ALL_MONTHS)
lat_n=((lat_grid-lat_grid.mean())/lat_grid.std()).flatten()
lon_n=((lon_grid-lon_grid.mean())/lon_grid.std()).flatten()
cl_feats.extend([lat_n,lon_n])
X_cl=np.stack(cl_feats,axis=1); X_cl=(X_cl-X_cl.mean(0))/(X_cl.std(0)+1e-8)

km=KMeans(n_clusters=K_CLUSTERS,n_init=20,random_state=42)
cluster_map=km.fit_predict(X_cl).reshape(GRID_H,GRID_W).astype(np.int32)

def norm01(x): return (x-x.min())/(x.max()-x.min()+1e-8)
means=np.array([pm25_mean[cluster_map==k].mean() for k in range(K_CLUSTERS)])
epfs =np.array([ep_freq[cluster_map==k].mean()   for k in range(K_CLUSTERS)])
stds =np.array([np.sqrt(pm25_var[cluster_map==k].mean()) for k in range(K_CLUSTERS)])
cluster_weights=0.5+2.5*norm01(0.4*norm01(means)+0.4*norm01(epfs)+0.2*norm01(stds))

cluster_map_norm=(cluster_map.astype(np.float32)-K_CLUSTERS/2)/(K_CLUSTERS/2)
print(f'Clusters done. Weights: [{cluster_weights.min():.3f}, {cluster_weights.max():.3f}]')


## 2. Normalization Stats
*Computed from training data exactly as during training.*

In [ ]:
def compute_norm_stats(all_months, features, train_hours, log_candidates, unit_scale):
    stats={}
    for feat in features:
        vals=[]
        for month in all_months:
            d=load_feature(month,feat)
            if d is None: continue
            chunk=np.maximum(d[:train_hours].flatten(),0)*unit_scale.get(feat,1.0); vals.append(chunk)
        if not vals: stats[feat]={'use_log':False,'norm_type':'zscore','loc':0.,'scale':1.}; continue
        raw=np.concatenate(vals); rs=scipy_skew(raw); lv=np.log1p(raw); ls=scipy_skew(lv)
        use_log=feat in log_candidates and abs(ls)<abs(rs)*0.8
        work=lv if use_log else raw; std=work.std()
        if std<0.01:
            p1=float(np.percentile(work,1)); p99=float(np.percentile(work,99))
            stats[feat]={'use_log':use_log,'norm_type':'percentile','loc':p1,'scale':max(p99-p1,1e-6)}
        else:
            stats[feat]={'use_log':use_log,'norm_type':'zscore','loc':float(work.mean()),'scale':float(std)+1e-8}
        del raw,vals; gc.collect()
    return stats

print('Computing normalization stats...')
norm_stats = compute_norm_stats(ALL_MONTHS, ALL_FEAT, TRAIN_HOURS, LOG_CANDIDATES, UNIT_SCALE)
print('✓ Norm stats computed')


## 3. Episode Frequency Prior
*Used as a static input channel to the model.*

In [ ]:
print('Computing episode frequency prior...')
ep_masks = {}
for month in ALL_MONTHS:
    d = load_feature(month, 'cpm25')
    ep_masks[month] = detect_episodes_grid(d)
    del d; gc.collect()

ep_freq_prior = np.mean([ep_masks[m].mean(0) for m in ALL_MONTHS], 0)
ep_freq_prior = (ep_freq_prior - ep_freq_prior.mean()) / (ep_freq_prior.std() + 1e-8)
print(f'Ep-freq prior: [{ep_freq_prior.min():.2f}, {ep_freq_prior.max():.2f}] ✓')


## 4. Model Definition

In [ ]:
class ConvBNGELU(nn.Module):
    def __init__(self,ic,oc,k=3,p=1):
        super().__init__()
        self.b=nn.Sequential(nn.Conv2d(ic,oc,k,padding=p,bias=False),nn.BatchNorm2d(oc),nn.GELU(),
                             nn.Conv2d(oc,oc,k,padding=p,bias=False),nn.BatchNorm2d(oc),nn.GELU())
    def forward(self,x): return self.b(x)

class ResBlock(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.c=nn.Sequential(nn.Conv2d(ch,ch,3,padding=1,bias=False),nn.BatchNorm2d(ch),nn.GELU(),
                             nn.Conv2d(ch,ch,3,padding=1,bias=False),nn.BatchNorm2d(ch))
    def forward(self,x): return F.gelu(x+self.c(x))

class ConvLSTMCell(nn.Module):
    def __init__(self,id_,hd,ks=3):
        super().__init__(); self.hd=hd
        self.conv=nn.Conv2d(id_+hd,4*hd,ks,padding=ks//2,bias=True)
    def forward(self,x,hc):
        h,c=hc; i,f,o,g=self.conv(torch.cat([x,h],1)).chunk(4,1)
        c2=torch.sigmoid(f)*c+torch.sigmoid(i)*torch.tanh(g)
        return torch.sigmoid(o)*torch.tanh(c2),c2
    def init(self,B,H,W,dev): z=torch.zeros(B,self.hd,H,W,device=dev); return z,z

class ConvLSTM(nn.Module):
    def __init__(self,id_,hd,nl=1):
        super().__init__()
        self.cells=nn.ModuleList([ConvLSTMCell(id_ if l==0 else hd,hd) for l in range(nl)])
    def forward(self,seq):
        B,T,C,H,W=seq.shape; hc=[c.init(B,H,W,seq.device) for c in self.cells]
        for t in range(T):
            inp=seq[:,t]
            for l,cell in enumerate(self.cells): h_,c_=cell(inp,hc[l]); hc[l]=(h_,c_); inp=h_
        return hc[-1][0]

class SimplifiedPM25Net(nn.Module):
    def __init__(self,in_ch,base_ch=24,lstm_dim=48,n_levels=2,forecast=16,lstm_layers=1):
        super().__init__(); self.forecast=forecast
        self.temporal=ConvLSTM(in_ch,lstm_dim,nl=lstm_layers)
        self.t_proj=nn.Sequential(nn.Conv2d(lstm_dim,base_ch,1,bias=False),nn.BatchNorm2d(base_ch),nn.GELU())
        self.enc,self.pool,self.enc_ch=nn.ModuleList(),nn.ModuleList(),[]
        ch=base_ch
        for lvl in range(n_levels):
            oc=base_ch*(2**lvl); self.enc.append(ConvBNGELU(ch,oc))
            self.pool.append(nn.MaxPool2d(2)); self.enc_ch.append(oc); ch=oc
        bot=base_ch*(2**n_levels)
        self.bot=nn.Sequential(ConvBNGELU(ch,bot),ResBlock(bot),ResBlock(bot))
        self.up,self.dec=nn.ModuleList(),nn.ModuleList()
        ch=bot
        for lvl in reversed(range(n_levels)):
            sc=self.enc_ch[lvl]; self.up.append(nn.ConvTranspose2d(ch,sc,2,2))
            self.dec.append(ConvBNGELU(sc*2,sc)); ch=sc
        self.head=nn.Sequential(nn.Conv2d(ch,ch,3,padding=1,bias=False),nn.BatchNorm2d(ch),
                                 nn.GELU(),nn.Conv2d(ch,forecast,1))
    def forward(self,x):
        pers=x[:,-1,0:1]; h=self.temporal(x); feat=self.t_proj(h); skips=[]
        for enc,pool in zip(self.enc,self.pool): feat=enc(feat); skips.append(feat); feat=pool(feat)
        feat=self.bot(feat)
        for up,dec,skip in zip(self.up,self.dec,reversed(skips)):
            feat=up(feat)
            if feat.shape[-2:]!=skip.shape[-2:]:
                feat=F.interpolate(feat,skip.shape[-2:],mode='bilinear',align_corners=False)
            feat=dec(torch.cat([feat,skip],1))
        return pers+self.head(feat)


## 5. Load Checkpoint

In [ ]:
model = SimplifiedPM25Net(
    in_ch=N_CHANNELS, base_ch=BASE_CHANNELS, lstm_dim=CONVLSTM_DIM,
    n_levels=N_LEVELS, forecast=FORECAST, lstm_layers=LSTM_LAYERS
).to(DEVICE)

state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {CHECKPOINT_PATH}')
print(f'Parameters: {n_params:,} ({n_params/1e6:.2f}M)')

# Sanity check forward pass
with torch.no_grad():
    dummy = torch.randn(1, LOOKBACK, N_CHANNELS, GRID_H, GRID_W, device=DEVICE)
    out = model(dummy)
    assert out.shape == (1, FORECAST, GRID_H, GRID_W), f'Unexpected output shape: {out.shape}'
    print(f'Forward pass: {dummy.shape} → {out.shape} ✓')
del dummy, out


## 6. Load Test Data + Run Inference (TTA: E-W flip)

In [ ]:
def normalize(arr, feat):
    s=norm_stats[feat]; arr=np.maximum(arr.astype(np.float32),0)*UNIT_SCALE.get(feat,1.0)
    if s['use_log']: arr=np.log1p(arr)
    return (arr-s['loc'])/s['scale']

def denorm_pm25(arr):
    s=norm_stats['cpm25']; out=arr*s['scale']+s['loc']
    if s['use_log']: out=np.expm1(np.clip(out,-10,20))
    return np.maximum(out,0)

def load_test_tensor(test_dir):
    channels=[]; ref_shape=None
    for feat in ALL_FEAT:
        p=test_dir/f'{feat}.npy'
        arr=np.load(p).astype(np.float32) if p.exists() else np.zeros(ref_shape,dtype=np.float32)
        if ref_shape is None: ref_shape=arr.shape
        s=norm_stats[feat]
        arr=np.maximum(arr,0)*UNIT_SCALE.get(feat,1.0)
        if s['use_log']: arr=np.log1p(arr)
        arr=(arr-s['loc'])/s['scale']
        channels.append(arr)
    N,T,H,W=ref_shape
    sin_ch=np.zeros((N,T,H,W),np.float32); cos_ch=np.ones((N,T,H,W),np.float32)
    tp=test_dir/'time.npy'
    if tp.exists():
        try:
            times=np.load(tp); hh=times%24
            sin_ch=(np.sin(2*np.pi*hh/24)[:,:,None,None]*np.ones((N,T,H,W))).astype(np.float32)
            cos_ch=(np.cos(2*np.pi*hh/24)[:,:,None,None]*np.ones((N,T,H,W))).astype(np.float32)
            print('  time.npy hour encoding ✓')
        except: pass
    ep_ch=np.broadcast_to(ep_freq_prior[None,None],(N,T,H,W)).copy().astype(np.float32)
    cl_ch=np.broadcast_to(cluster_map_norm[None,None],(N,T,H,W)).copy().astype(np.float32)
    channels.extend([sin_ch,cos_ch,ep_ch,cl_ch])
    tensor=np.stack(channels,axis=2).astype(np.float32)  # (N, T, C, H, W)
    print(f'Test tensor: {tensor.shape} ✓')
    return tensor

print('Loading test data...')
test_tensor = load_test_tensor(TEST_DIR)
N_test = test_tensor.shape[0]
assert test_tensor.shape[2] == N_CHANNELS, f'Channel mismatch: got {test_tensor.shape[2]}, expected {N_CHANNELS}'

# TTA: average original + E-W flipped prediction
print(f'Running inference on {N_test} test samples (TTA: E-W flip)...')
p_orig, p_flip = [], []
with torch.no_grad():
    for s in range(0, N_test, 8):
        e = min(s+8, N_test)
        b = torch.from_numpy(test_tensor[s:e]).float().to(DEVICE)
        p_orig.append(denorm_pm25(model(b).cpu().numpy()))
        p_flip.append(denorm_pm25(model(b.flip(-1)).cpu().numpy())[...,::-1].copy())
        print(f'  {e}/{N_test}', end='\r')

preds = 0.5*np.concatenate(p_orig, 0) + 0.5*np.concatenate(p_flip, 0)
print(f'\nInference done. Raw predictions: {preds.shape}')


## 7. Post-process + Save

In [ ]:
# Clip to 1.25× training max and handle NaNs
train_max = max(float(load_feature(m,'cpm25').max()) for m in ALL_MONTHS)
print(f'Training PM2.5 max: {train_max:.1f} µg/m³  →  clip at {train_max*1.25:.1f}')

preds = np.clip(preds, 0, train_max * 1.25)
preds = np.nan_to_num(preds, nan=0.)

# Reshape to submission format: (N, H, W, T)
preds_submit = preds.transpose(0, 2, 3, 1)
assert preds_submit.shape == (218, 140, 124, 16), f'Wrong shape: {preds_submit.shape}'

out_path = OUTPUT / 'preds.npy'
np.save(out_path, preds_submit)

# Verify
loaded = np.load(out_path)
assert loaded.shape == (218, 140, 124, 16)
print(f'\nSaved: {out_path}')
print(f'Shape: {loaded.shape}  ✓')
print(f'Range: [{preds_submit.min():.1f}, {preds_submit.max():.1f}] µg/m³')
print(f'NaNs:  {np.isnan(preds_submit).sum()}')
print('\n✓ Ready to submit!')
